In [ ]:
#!/usr/bin/env python3
"""
Toy ablation line plots for S3, S4, and S5.

Metrics included:
1. Forward MLP model MAE
2. Inverse MLP test accuracy

Expected input:
<EVALUATION_ROOT>/<dataset>_pseudo_pairing_evaluation/<group>/
    result_analysis/selected_variants_TEMPLATE_EDIT_ME.csv

Ablation encoding:
S3: x = number of metacells averaged (k); color = total metacells
S5: x = OT top-k; color = total metacells
S4: no k exists, so x = total metacells and one line is drawn

All S3/S4/S5 rows are used by default; select_for_final is ignored unless
USE_ONLY_SELECT_FOR_FINAL_ROWS=True.
"""

from __future__ import annotations

import re
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ============================================================
# User configuration
# ============================================================

EVALUATION_ROOT = Path("Perturbation/evaluation")

DATASET_NAMES = [
    "Replogle_K562_essential",
    "Replogle_RPE",
    "NormanWeissman2019",
    "ChangYe",
    "ZhaoSims2021",
]

PERTURBED_GROUPS = ["single", "dual", "multi"]

RUN_ALL_VALID_PATHS = True
SELECTED_SUB_PATH: Path | None = None
SELECT_CSV_NAME = "selected_variants_TEMPLATE_EDIT_ME.csv"

# Ablation uses every S3/S4/S5 configuration, not only final selections.
USE_ONLY_SELECT_FOR_FINAL_ROWS = False

STRATEGIES_TO_PLOT = [
    "S3_SEACell_metacell_average",
    "S4_SEACell_balanced_random_sample",
    "S5_SEACell_OT_sampled_average",
]


OUTPUT_FOLDER_NAME = "_ablation_S3_S4_S5_all_metrics"
OUTPUT_ROOT = EVALUATION_ROOT / OUTPUT_FOLDER_NAME


METRICS: dict[str, dict[str, Any]] = {
    "control_local_mixing": {
        "mean_column": "control_manifold__control_pseudo_local_mixing_score_mean",
        "std_column": "control_manifold__control_pseudo_local_mixing_score_std",
        "n_column": "control_manifold__control_pseudo_local_mixing_score_n",
        "ylabel": "Local mixing score",
        "title": "Control–pseudo-control local mixing",
        "direction": "higher",
    },
    "perturbation_effect_pearson": {
        "mean_column": "perturbation_effect__perturbation_effect_pearson_mean",
        "std_column": "perturbation_effect__perturbation_effect_pearson_std",
        "n_column": "perturbation_effect__perturbation_effect_pearson_n",
        "ylabel": "Perturbation-effect Pearson",
        "title": "Perturbation-effect Pearson correlation",
        "direction": "higher",
    },
    "forward_model_mse": {
        "mean_column": "mlp_forward__model_mse_mean",
        "std_column": "mlp_forward__model_mse_std",
        "n_column": "mlp_forward__model_mse_n",
        "ylabel": "Forward MLP model MSE",
        "title": "Forward perturbation prediction MSE",
        "direction": "lower",
    },
    "forward_model_mae": {
        "mean_column": "mlp_forward__model_mae_mean",
        "std_column": "mlp_forward__model_mae_std",
        "n_column": "mlp_forward__model_mae_n",
        "ylabel": "Forward MLP model MAE",
        "title": "Forward perturbation prediction MAE",
        "direction": "lower",
    },
    "inverse_test_accuracy": {
        "mean_column": "mlp_inverse__test_accuracy_mean",
        "std_column": "mlp_inverse__test_accuracy_std",
        "n_column": "mlp_inverse__test_accuracy_n",
        "ylabel": "Inverse MLP test accuracy",
        "title": "Inverse perturbation-identity accuracy",
        "direction": "higher",
    },
    "inverse_macro_auc": {
        "mean_column": "mlp_inverse__macro_auc_mean",
        "std_column": "mlp_inverse__macro_auc_std",
        "n_column": "mlp_inverse__macro_auc_n",
        "ylabel": "Inverse MLP macro AUC",
        "title": "Inverse perturbation-identity macro AUC",
        "direction": "higher",
    },
}

INDIVIDUAL_FIGSIZE = (10, 6)

LINEWIDTH = 1.8
MARKER = "o"
MARKER_SIZE = 6.5
CAPSIZE = 4
ERRORBAR_LINEWIDTH = 1.0
ALPHA = 0.92
GRID_ALPHA = 0.25

DPI = 300
SAVE_PNG = True
SAVE_SVG = True
SHOW_FIGURES = False

MIN_RELATIVE_Y_SPAN = 0.08
MIN_ABSOLUTE_Y_SPAN = 1e-4
ACCURACY_MIN_SPAN = 0.05

# Colors assigned to k values in ascending order within each strategy.
# S3 normally maps k = 3, 5, 10 to these colors.
# S5 normally maps k = 1, 3, 5 to these colors.
K_LINE_COLORS = ["#b7d8f6", "#88b6f0", "#293e62"]

# S4 has no k parameter, so use the darkest requested color for its single line.
S4_LINE_COLOR = "#293e62"

STRATEGY_SHORT_NAMES = {
    "S3_SEACell_metacell_average": "S3: Random metacell average",
    "S4_SEACell_balanced_random_sample": "S4: Metacell balanced random",
    "S5_SEACell_OT_sampled_average": "S5: Metacell OT sampled average",
}


# ============================================================
# Data structures and IO
# ============================================================

@dataclass
class DatasetGroupInput:
    dataset: str
    group: str
    sub_path: Path
    csv_path: Path
    dataframe: pd.DataFrame


def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".tsv", ".txt"}:
        return pd.read_csv(path, sep=None, engine="python")
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported table format: {path}")


def as_bool_series(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})


def discover_dataset_group_inputs() -> list[DatasetGroupInput]:
    records: list[DatasetGroupInput] = []

    if RUN_ALL_VALID_PATHS:
        candidates: list[tuple[str, str, Path]] = []
        for dataset in DATASET_NAMES:
            for group in PERTURBED_GROUPS:
                sub_path = EVALUATION_ROOT / f"{dataset}_pseudo_pairing_evaluation" / group
                candidates.append((dataset, group, sub_path))
    else:
        if SELECTED_SUB_PATH is None:
            raise ValueError("RUN_ALL_VALID_PATHS=False requires SELECTED_SUB_PATH.")
        sub_path = Path(SELECTED_SUB_PATH)
        dataset = sub_path.parent.name.replace("_pseudo_pairing_evaluation", "")
        group = sub_path.name
        candidates = [(dataset, group, sub_path)]

    for dataset, group, sub_path in candidates:
        csv_path = sub_path / "result_analysis" / SELECT_CSV_NAME
        if not csv_path.exists():
            continue
        try:
            df = read_table(csv_path)
        except Exception as exc:
            warnings.warn(f"Failed to read {csv_path}: {exc}")
            continue
        if df.empty:
            warnings.warn(f"Skipping empty table: {csv_path}")
            continue
        records.append(DatasetGroupInput(dataset, group, sub_path, csv_path, df))

    if not records:
        raise FileNotFoundError(
            "No valid dataset/group tables were found under "
            f"{EVALUATION_ROOT}. Expected files named {SELECT_CSV_NAME}."
        )
    return records


# ============================================================
# Parameter extraction
# ============================================================

def numeric_series(df: pd.DataFrame, candidates: Iterable[str]) -> pd.Series:
    result = pd.Series(np.nan, index=df.index, dtype=float)
    for column in candidates:
        if column not in df.columns:
            continue
        values = pd.to_numeric(df[column], errors="coerce")
        result = result.where(result.notna(), values)
    return result


def extract_regex_numeric(variant_ids: pd.Series, patterns: Iterable[str]) -> pd.Series:
    result = pd.Series(np.nan, index=variant_ids.index, dtype=float)
    text = variant_ids.fillna("").astype(str)
    for pattern in patterns:
        extracted = text.str.extract(pattern, expand=False)
        values = pd.to_numeric(extracted, errors="coerce")
        result = result.where(result.notna(), values)
    return result


def infer_strategy(df: pd.DataFrame) -> pd.Series:
    strategy = (
        df["strategy"].fillna("").astype(str)
        if "strategy" in df.columns
        else pd.Series("", index=df.index, dtype=object)
    )
    if "variant_id" not in df.columns:
        return strategy

    variant = df["variant_id"].fillna("").astype(str)
    inferred = pd.Series("", index=df.index, dtype=object)
    for known in STRATEGIES_TO_PLOT:
        inferred = inferred.where(~variant.str.startswith(known), known)
    return strategy.where(strategy.ne(""), inferred)


def prepare_ablation_table(source: DatasetGroupInput) -> pd.DataFrame:
    df = source.dataframe.copy()
    if "variant_id" not in df.columns:
        raise KeyError(f"'variant_id' is missing from {source.csv_path}")

    df["strategy"] = infer_strategy(df)

    if USE_ONLY_SELECT_FOR_FINAL_ROWS and "select_for_final" in df.columns:
        df = df[as_bool_series(df["select_for_final"])].copy()

    df = df[df["strategy"].isin(STRATEGIES_TO_PLOT)].copy()
    if df.empty:
        return df

    variant_id = df["variant_id"].fillna("").astype(str)

    df["ablation_n_metacells"] = numeric_series(
        df,
        ["n_metacells", "n_metacells_requested", "n_metacells_observed"],
    )
    nmc_from_id = extract_regex_numeric(variant_id, [r"__nmc_(\d+)", r"/nmc_(\d+)"])
    df["ablation_n_metacells"] = df["ablation_n_metacells"].where(
        df["ablation_n_metacells"].notna(), nmc_from_id
    )

    # S3: k is the number of metacells averaged.
    df["ablation_s3_k"] = numeric_series(
        df,
        ["n_metacells_to_average", "sampled_metacells_k"],
    )
    s3_k_from_id = extract_regex_numeric(
        variant_id,
        [r"__sampledMC_(\d+)", r"__k_(\d+)", r"/k_(\d+)"],
    )
    df["ablation_s3_k"] = df["ablation_s3_k"].where(
        df["ablation_s3_k"].notna(), s3_k_from_id
    )

    # S5: k is the OT top-k.
    df["ablation_s5_k"] = numeric_series(df, ["top_k", "top_k_metacells"])
    s5_k_from_id = extract_regex_numeric(
        variant_id,
        [r"__topk_(\d+)", r"/topk_0*(\d+)"],
    )
    df["ablation_s5_k"] = df["ablation_s5_k"].where(
        df["ablation_s5_k"].notna(), s5_k_from_id
    )

    df["dataset_id_for_plot"] = source.dataset
    df["perturbed_group_for_plot"] = source.group
    df["source_csv"] = str(source.csv_path)

    # The x-axis is always the total number of metacells.
    df["ablation_x"] = df["ablation_n_metacells"]

    # Standardized line-group parameter. S4 has no k value.
    df["ablation_k"] = np.nan
    s3 = df["strategy"].eq("S3_SEACell_metacell_average")
    s5 = df["strategy"].eq("S5_SEACell_OT_sampled_average")
    df.loc[s3, "ablation_k"] = df.loc[s3, "ablation_s3_k"]
    df.loc[s5, "ablation_k"] = df.loc[s5, "ablation_s5_k"]

    return df


# ============================================================
# Plot-data preparation
# ============================================================

def metric_plot_table(prepared_df: pd.DataFrame, strategy: str, metric_key: str) -> pd.DataFrame:
    info = METRICS[metric_key]
    mean_col = info["mean_column"]
    std_col = info["std_column"]
    n_col = info["n_column"]

    if mean_col not in prepared_df.columns:
        return pd.DataFrame()

    df = prepared_df[prepared_df["strategy"].eq(strategy)].copy()
    if df.empty:
        return df

    df["plot_mean"] = pd.to_numeric(df[mean_col], errors="coerce")
    df["plot_std"] = (
        pd.to_numeric(df[std_col], errors="coerce")
        if std_col in df.columns else np.nan
    )
    df["plot_n"] = (
        pd.to_numeric(df[n_col], errors="coerce")
        if n_col in df.columns else np.nan
    )
    df = df.dropna(subset=["plot_mean", "ablation_x"]).copy()

    if strategy in {"S3_SEACell_metacell_average", "S5_SEACell_OT_sampled_average"}:
        df = df.dropna(subset=["ablation_n_metacells", "ablation_k"]).copy()
        group_cols = [
            "dataset_id_for_plot",
            "perturbed_group_for_plot",
            "strategy",
            "ablation_k",
            "ablation_x",
        ]
    else:
        df = df.dropna(subset=["ablation_n_metacells"]).copy()
        group_cols = [
            "dataset_id_for_plot",
            "perturbed_group_for_plot",
            "strategy",
            "ablation_x",
        ]

    def pooled_std(values: pd.Series) -> float:
        arr = pd.to_numeric(values, errors="coerce").dropna().to_numpy(dtype=float)
        return float(np.sqrt(np.mean(arr ** 2))) if len(arr) else np.nan

    out = (
        df.groupby(group_cols, dropna=False, as_index=False)
        .agg(
            plot_mean=("plot_mean", "mean"),
            plot_std=("plot_std", pooled_std),
            plot_n=("plot_n", "sum"),
            variant_id=("variant_id", lambda x: " | ".join(sorted(set(map(str, x))))),
        )
        .sort_values(group_cols)
    )
    return out


# ============================================================
# Plotting helpers
# ============================================================

def x_axis_label(strategy: str) -> str:
    return "Number of metacells"


def build_k_color_map(plot_df: pd.DataFrame) -> dict[float, str]:
    """Assign the three requested colors to sorted k values."""
    if "ablation_k" not in plot_df.columns:
        return {}

    k_values = sorted(
        pd.to_numeric(plot_df["ablation_k"], errors="coerce")
        .dropna()
        .unique()
        .tolist()
    )

    if len(k_values) > len(K_LINE_COLORS):
        raise ValueError(
            f"Found {len(k_values)} k values ({k_values}), but only "
            f"{len(K_LINE_COLORS)} colors were provided."
        )

    return {
        float(k): K_LINE_COLORS[i]
        for i, k in enumerate(k_values)
    }


def stabilize_y_limits(ax: plt.Axes, metric_key: str, values: Iterable[float]) -> None:
    vals = np.asarray([float(v) for v in values if np.isfinite(v)], dtype=float)
    if len(vals) == 0:
        return

    y_min = float(vals.min())
    y_max = float(vals.max())
    center = 0.5 * (y_min + y_max)
    span = y_max - y_min

    bounded_unit_metrics = {
        "control_local_mixing",
        "inverse_test_accuracy",
        "inverse_macro_auc",
    }

    if metric_key in bounded_unit_metrics:
        min_span = ACCURACY_MIN_SPAN
    else:
        min_span = max(abs(center) * MIN_RELATIVE_Y_SPAN, MIN_ABSOLUTE_Y_SPAN)

    if span < min_span:
        y_min = center - min_span / 2
        y_max = center + min_span / 2
        span = min_span

    pad = max(span * 0.12, min_span * 0.06)
    y_min -= pad
    y_max += pad

    if metric_key in {
        "control_local_mixing",
        "inverse_test_accuracy",
        "inverse_macro_auc",
    }:
        y_min = max(0.0, y_min)
        y_max = min(1.02, y_max)

    if metric_key == "perturbation_effect_pearson":
        y_min = max(-1.02, y_min)
        y_max = min(1.02, y_max)
    ax.set_ylim(y_min, y_max)


def plot_strategy_metric_on_axis(
    ax: plt.Axes,
    plot_df: pd.DataFrame,
    strategy: str,
    metric_key: str,
    panel_title: str,
    show_legend: bool,
) -> None:
    info = METRICS[metric_key]
    y_extent: list[float] = []

    if strategy in {"S3_SEACell_metacell_average", "S5_SEACell_OT_sampled_average"}:
        k_color_map = build_k_color_map(plot_df)

        for k_value, line_df in plot_df.groupby("ablation_k", sort=True):
            line_df = line_df.sort_values("ablation_x")
            x = line_df["ablation_x"].to_numpy(dtype=float)
            y = line_df["plot_mean"].to_numpy(dtype=float)
            yerr = line_df["plot_std"].fillna(0).to_numpy(dtype=float)

            k_numeric = float(k_value)
            color = k_color_map[k_numeric]
            k_label = int(k_numeric) if k_numeric.is_integer() else k_numeric

            ax.errorbar(
                x,
                y,
                yerr=yerr,
                label=f"k = {k_label}",
                color=color,
                marker=MARKER,
                markeredgecolor="#202020",
                markeredgewidth=0.6,
                markersize=MARKER_SIZE,
                linewidth=LINEWIDTH,
                elinewidth=ERRORBAR_LINEWIDTH,
                capsize=CAPSIZE,
                alpha=ALPHA,
                zorder=3,
            )

            y_extent.extend((y - yerr).tolist())
            y_extent.extend((y + yerr).tolist())
    else:
        # S4 has no k parameter, so it is represented by a single line.
        line_df = plot_df.sort_values("ablation_x")
        x = line_df["ablation_x"].to_numpy(dtype=float)
        y = line_df["plot_mean"].to_numpy(dtype=float)
        yerr = line_df["plot_std"].fillna(0).to_numpy(dtype=float)

        ax.errorbar(
            x,
            y,
            yerr=yerr,
            color=S4_LINE_COLOR,
            marker=MARKER,
            markeredgecolor="#202020",
            markeredgewidth=0.6,
            markersize=MARKER_SIZE,
            linewidth=LINEWIDTH,
            elinewidth=ERRORBAR_LINEWIDTH,
            capsize=CAPSIZE,
            alpha=ALPHA,
            zorder=3,
        )

        y_extent.extend((y - yerr).tolist())
        y_extent.extend((y + yerr).tolist())

    unique_x = sorted(
        pd.to_numeric(plot_df["ablation_x"], errors="coerce")
        .dropna()
        .unique()
    )
    ax.set_xticks(unique_x)
    ax.set_xlabel(x_axis_label(strategy))
    ax.set_ylabel(info["ylabel"])
    ax.set_title(panel_title)
    ax.grid(axis="both", alpha=GRID_ALPHA, linewidth=0.7, zorder=0)

    # Publication-style axes: retain only the left and bottom spines.
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    stabilize_y_limits(ax, metric_key, y_extent)

    if (
        show_legend
        and strategy
        in {"S3_SEACell_metacell_average", "S5_SEACell_OT_sampled_average"}
    ):
        n_k_values = int(
            pd.to_numeric(plot_df["ablation_k"], errors="coerce")
            .dropna()
            .nunique()
        )
        ax.legend(
            title="k value",
            frameon=False,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.17),
            ncol=max(1, min(3, n_k_values)),
            borderaxespad=0.0,
            columnspacing=1.6,
            handletextpad=0.6,
        )


def save_figure(fig: plt.Figure, base_path: Path) -> None:
    """Save one figure in the requested formats."""
    base_path.parent.mkdir(parents=True, exist_ok=True)

    if SAVE_PNG:
        fig.savefig(
            base_path.with_suffix(".png"),
            dpi=DPI,
            bbox_inches="tight",
        )

    if SAVE_SVG:
        fig.savefig(
            base_path.with_suffix(".svg"),
            bbox_inches="tight",
        )


def plot_individual_figure(
    plot_df: pd.DataFrame,
    dataset: str,
    group: str,
    strategy: str,
    metric_key: str,
) -> Path:
    fig, ax = plt.subplots(figsize=INDIVIDUAL_FIGSIZE)

    title = (
        f"{STRATEGY_SHORT_NAMES[strategy]}\n"
        f"{METRICS[metric_key]['title']}\n"
        f"{dataset} | {group}"
    )

    plot_strategy_metric_on_axis(
        ax=ax,
        plot_df=plot_df,
        strategy=strategy,
        metric_key=metric_key,
        panel_title=title,
        show_legend=True,
    )

    # Reserve space below the axes for the horizontal k-value legend.
    if strategy in {
        "S3_SEACell_metacell_average",
        "S5_SEACell_OT_sampled_average",
    }:
        fig.tight_layout(rect=(0.0, 0.12, 1.0, 1.0))
        fig.subplots_adjust(bottom=0.23)
    else:
        fig.tight_layout()

    output_base = (
        OUTPUT_ROOT
        / dataset
        / group
        / strategy
        / metric_key
    )
    save_figure(fig, output_base)

    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)

    return output_base


# ============================================================
# Main
# ============================================================

def main() -> None:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    sources = discover_dataset_group_inputs()
    n_figures = 0
    manifest_rows: list[dict[str, Any]] = []

    for source in sources:
        try:
            prepared = prepare_ablation_table(source)
        except Exception as exc:
            warnings.warn(f"Failed to prepare {source.csv_path}: {exc}")
            continue

        if prepared.empty:
            continue

        for strategy in STRATEGIES_TO_PLOT:
            for metric_key, metric_info in METRICS.items():
                plot_df = metric_plot_table(
                    prepared_df=prepared,
                    strategy=strategy,
                    metric_key=metric_key,
                )

                if plot_df.empty:
                    warnings.warn(
                        f"Skipping missing metric/strategy combination: "
                        f"{source.dataset} | {source.group} | "
                        f"{strategy} | {metric_key}"
                    )
                    continue

                output_base = plot_individual_figure(
                    plot_df=plot_df,
                    dataset=source.dataset,
                    group=source.group,
                    strategy=strategy,
                    metric_key=metric_key,
                )

                manifest_rows.append(
                    {
                        "dataset": source.dataset,
                        "group": source.group,
                        "strategy": strategy,
                        "metric": metric_key,
                        "metric_mean_column": metric_info["mean_column"],
                        "source_csv": str(source.csv_path),
                        "output_png": (
                            str(output_base.with_suffix(".png"))
                            if SAVE_PNG else ""
                        ),
                        "output_svg": (
                            str(output_base.with_suffix(".svg"))
                            if SAVE_SVG else ""
                        ),
                        "n_parameter_points": int(plot_df.shape[0]),
                    }
                )
                n_figures += 1

    if n_figures == 0:
        raise RuntimeError(
            "No figures were generated because no usable S3/S4/S5 metric rows "
            "were found."
        )

    manifest = pd.DataFrame(manifest_rows)
    manifest.to_csv(
        OUTPUT_ROOT / "ablation_figure_manifest.csv",
        index=False,
    )

    print("=" * 80)
    print(f"Saved {n_figures} individual ablation figures.")
    print(f"Output root: {OUTPUT_ROOT}")
    print("Folder structure:")
    print("  <dataset>/<group>/<strategy>/<metric>.png|svg")
    print("Line encoding:")
    print("  x-axis: total number of metacells")
    print("  S3/S5: separate lines for different k values")
    print(f"  k colors: {K_LINE_COLORS}")
    print("  S4: one line because no k parameter exists")
    print("=" * 80)


if __name__ == "__main__":
    main()
